# Lab 5: Recomendadores con ALS y búsqueda semántica con embeddings

**Análisis de Big Data · Magíster en Data Science UDD · Sesión 5 (viernes 11 de septiembre)**

Este laboratorio cambia de dataset por primera vez en el curso: **MovieLens pequeño**
(`ml-latest-small`: 610 usuarios, 9.742 películas, 100.836 calificaciones). La razón es de método,
no de comodidad: el objetivo de aprendizaje es el **criterio**: qué cambia al mover `rank`, qué
pasa con un usuario nuevo, cuándo el contenido complementa a la interacción. El criterio se
aprende iterando. Con 100 mil calificaciones cada ajuste toma segundos; en 20 minutos caben tres
configuraciones comparadas.

Las dos mitades trabajan sobre **el mismo catálogo**: **(a)** filtrado colaborativo
(*collaborative filtering*) con ALS y tracking en MLflow; **(b)** representaciones vectoriales
(*embeddings*) de título y género con `sentence-transformers` y búsqueda semántica. La misma
película recomendada por interacción y por contenido, y el usuario sin historial resuelto por la
segunda vía.

---

## Qué se entrega el miércoles 16 de septiembre, 23:59

**No se entrega el notebook completo.** Se entrega:

1. La **tabla de `mlflow.search_runs()`** con **al menos tres configuraciones de ALS** y su RMSE
   (sección 10).
2. El **top-5 de una consulta semántica propia** (sección 9, celda `MI_CONSULTA`).
3. **Un párrafo** que relacione ambas: qué le recomendaría ALS a un usuario nuevo y por qué la
   búsqueda por contenido cubre ese caso (sección 10b, celda de texto a completar).

El **Quiz 2 es el lunes 14**: el fin de
semana es para el quiz; el laboratorio vence después, a propósito.

---

## Objetivos

- Entrenar un recomendador por factorización de matrices con `ALS` y entender qué se distribuye
  entre ejecutores y por qué el algoritmo alterna.
- Reconocer el **arranque en frío** (*cold start*) en su forma más concreta, un `NaN` en el RMSE,
  y la decisión que lo resuelve.
- Medir el **sesgo de popularidad** con la cobertura del catálogo, no con la intuición.
- Calcular embeddings **a escala** con una función definida por el usuario de pandas
  (*pandas UDF*) de tipo iterador: el modelo se carga una vez por partición, no una vez por fila.
- Comparar las tres configuraciones de ALS **en la misma tabla** de MLflow, con métrica y tiempo:
  nuevamente aparece `search_runs()`, que será una exigencia en la Fase 2.

## Entrega actualizada y trazabilidad

Se preservan todas las celdas de código y salidas de la ejecución de Dante. Esta actualización documental no volvió a entrenar modelos. La base MLflow se consultó en modo solo lectura el 2026-09-18T22:12:33.607973+00:00.

**Selección:** última corrida finalizada de cada una de las tres configuraciones ALS del laboratorio pequeño, según `start_time`. Son las corridas que concuerdan con las salidas base de esta ejecución. No se promediaron tiempos ni se eligió la corrida más rápida.

| experimento   | run_id                           |   RMSE |   cobertura_% |   seg_entrenamiento |
|:--------------|:---------------------------------|-------:|--------------:|--------------------:|
| als_base      | ba9f050134b14ae991590295e2937068 | 0.8814 |          6.83 |              1.4534 |
| als_rank_alto | 9fbf0f8caf9b463bb36ace59e72001a0 | 0.8736 |         12.95 |              2.3041 |
| als_reg_alta  | dd46f64db2bf4f148066df10a084a37d | 0.9978 |          0.32 |              1.4713 |

**Consulta propia: Ciberpunk.** La similitud es un puntaje geométrico, no probabilidad de agrado.

|   movieId | titulo                                          |   similitud |
|----------:|:------------------------------------------------|------------:|
|    160684 | Marauders (2016)                                |       0.388 |
|     72165 | Cirque du Freak: The Vampire’s Assistant (2009) |       0.366 |
|    168174 | Jim Gaffigan: Cinco (2017)                      |       0.345 |
|      1675 | Incognito (1997)                                |       0.342 |
|     88356 | Smurfs, The (2011)                              |       0.339 |

El top-5 original se conserva como evidencia de una búsqueda poco pertinente. E4 evalúa otra representación; sus resultados no reemplazan silenciosamente esta consulta de la entrega.

**Registro completo:** 18 runs del experimento `lab5_recomendadores`, incluidos históricos y extensiones. La sección 10 original conserva su salida de 14 runs, porque se ejecutó antes de finalizar las extensiones. El anexo de registro actualizado y los CSV de este paquete contienen los 18 runs disponibles al exportar. E3 se midió en el notebook, pero no creó un run de MLflow; se exporta como evidencia separada.

**Estado:** E1–E5 tienen resultados guardados. No quedan pendientes de ejecución en esta versión; sí queda la revisión crítica de pertinencia, comparabilidad de tiempos y límites de generalización. Los tiempos son de ejecuciones individuales, no un benchmark con repeticiones controladas.


## 0. Setup

Misma receta del Lab 4 más una instalación nueva: `sentence-transformers`, que trae el modelo de
embeddings. Demora un par de minutos.

Los *runs* se leen con `mlflow.search_runs()`.

In [ ]:
# Colab. En una instalación local con PySpark ya presente, omitir el bloque de instalación.
import sys
print("Python", ".".join(map(str, sys.version_info[:3])))

# Java 17 es compatible con PySpark 3.5.1. En macOS Apple Silicon se instala con Homebrew
# solo si no existe una instalación compatible; el dataset y su ruta no cambian.
import os, platform, shutil, subprocess

JAVA_HOME_CANDIDATOS = [
    os.environ.get("JAVA_HOME"),
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/usr/lib/jvm/java-17-openjdk-amd64",
]

java_home = next((ruta for ruta in JAVA_HOME_CANDIDATOS if ruta and os.path.exists(ruta)), None)
if java_home is None and platform.system() == "Darwin" and shutil.which("brew"):
    subprocess.run(["brew", "install", "openjdk@17"], check=True)
    java_home = next((ruta for ruta in JAVA_HOME_CANDIDATOS if ruta and os.path.exists(ruta)), None)
if java_home is None:
    raise RuntimeError("No se encontró Java 17. Instálalo con 'brew install openjdk@17' y vuelve a ejecutar esta celda.")

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = os.path.join(java_home, "bin") + os.pathsep + os.environ["PATH"]
print("JAVA_HOME:", os.environ["JAVA_HOME"])
subprocess.run([os.path.join(java_home, "bin", "java"), "-version"], check=True)

# pyarrow solo como wheel; PySpark y MLflow fijados: la misma combinación que funcionó en el Lab 4.
!pip -q install --only-binary=:all: pyarrow
!pip -q install --prefer-binary pyspark==3.5.1 "mlflow>=2.16"
# Nuevo en este lab: el modelo de embeddings (usa el torch que Colab ya trae).
!pip -q install sentence-transformers

import time
import numpy as np, pandas as pd
import mlflow, mlflow.spark
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName("lab5-recomendadores")
         .config("spark.sql.shuffle.partitions", "16")      # dataset chico: menos particiones, menos overhead
         .config("spark.driver.memory", "6g")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| MLflow", mlflow.__version__)

In [ ]:
# La misma celda de tracking del Lab 4. Solo cambia el nombre del experimento.
RUTA_TRACKING = "/content" if os.access("/content", os.W_OK) else os.path.expanduser("~/lab5_recomendadores")
os.makedirs(RUTA_TRACKING, exist_ok=True)
RUTA_DB = os.path.join(RUTA_TRACKING, "mlflow.db")

mlflow.set_tracking_uri(f"sqlite:///{RUTA_DB}")
mlflow.set_experiment("lab5_recomendadores")
print("Tracking URI:", mlflow.get_tracking_uri())

# Alternativa si SQLite diera problemas: store de archivos, que exige el opt-out.
# import os; os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
# mlflow.set_tracking_uri(f"file:{os.path.join(RUTA_TRACKING, 'mlruns')}")

# Respaldo opcional al terminar: conserva los runs si se reinicia el entorno.
# from google.colab import drive; drive.mount("/content/drive")
# !cp {RUTA_DB} /content/drive/MyDrive/adbd/mlflow_lab5.db

## 1. Los datos: una matriz casi vacía

MovieLens pequeño se descarga directo de GroupLens (1 MB). Dos archivos importan:
`ratings.csv` (usuario, película, nota de 0,5 a 5, marca de tiempo) y `movies.csv` (título y géneros).

La cifra que ordena toda la primera mitad es la **densidad** de la matriz usuario–ítem: qué
fracción de las celdas tiene una calificación. El recomendador existe para llenar las demás.

In [ ]:
import os, ssl, shutil, zipfile, urllib.request

URL_ML = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
RUTA_DATOS = "/content" if os.access("/content", os.W_OK) else os.path.expanduser("~/lab5_recomendadores")
os.makedirs(RUTA_DATOS, exist_ok=True)
RUTA_ZIP = os.path.join(RUTA_DATOS, "ml-latest-small.zip")
RUTA_ML = os.path.join(RUTA_DATOS, "ml-latest-small")
ENLACE_RESPALDO = ""   # si nada de lo anterior funciona: enlace de Drive al mismo zip (receta gdown del Lab 4)

def zip_valido(ruta):
    """Un zip servido a medias o una página de error pesan poco y no abren como zip."""
    return os.path.exists(ruta) and os.path.getsize(ruta) > 100_000 and zipfile.is_zipfile(ruta)

def descargar(url, destino, verificar_ssl=True):
    contexto = None if verificar_ssl else ssl._create_unverified_context()
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(pedido, timeout=90, context=contexto) as r, open(destino, "wb") as f:
        shutil.copyfileobj(r, f)

if not os.path.exists(f"{RUTA_ML}/ratings.csv"):
    # El certificado de GroupLens caduca seguido: si la verificación falla, se reintenta sin ella.
    for etiqueta, kwargs in [("con verificación", {}), ("sin verificación de certificado", {"verificar_ssl": False})]:
        if zip_valido(RUTA_ZIP):
            break
        if os.path.exists(RUTA_ZIP):
            os.remove(RUTA_ZIP)
        try:
            descargar(URL_ML, RUTA_ZIP, **kwargs)
            print(f"descargado de GroupLens ({etiqueta}): {os.path.getsize(RUTA_ZIP)/1e6:.1f} MB")
        except Exception as e:
            print(f"intento {etiqueta}:", e)
    if not zip_valido(RUTA_ZIP) and ENLACE_RESPALDO:
        !gdown --fuzzy "$ENLACE_RESPALDO" -O "$RUTA_ZIP"
    if not zip_valido(RUTA_ZIP):
        raise RuntimeError("El zip no llegó completo. Pegar en ENLACE_RESPALDO el enlace de Drive al mismo ml-latest-small.zip y volver a ejecutar esta celda.")
    with zipfile.ZipFile(RUTA_ZIP) as z:
        z.extractall(RUTA_DATOS)

print("Datos en:", RUTA_ML)
print(sorted(os.listdir(RUTA_ML)))

In [ ]:
ratings = (spark.read.csv(f"{RUTA_ML}/ratings.csv", header=True, inferSchema=True)
           .select("userId", "movieId", "rating", "timestamp"))
movies  = spark.read.csv(f"{RUTA_ML}/movies.csv", header=True, inferSchema=True)

n_r = ratings.count()
n_u = ratings.select("userId").distinct().count()
n_m = movies.count()
print(f"calificaciones: {n_r:,} | usuarios: {n_u:,} | películas en catálogo: {n_m:,}")
print(f"densidad de la matriz usuario-ítem: {n_r / (n_u * n_m):.2%}  →  el {1 - n_r / (n_u * n_m):.1%} está vacío")

ratings.show(5)
movies.show(5, truncate=60)

## 2. El split, y una advertencia sobre el tiempo

Se usa `randomSplit` 80/20 con semilla fija, que es el protocolo estándar para comparar RMSE
entre configuraciones de ALS y el que permite que las tablas de distintos estudiantes sean
comparables.

Conviene decir lo que ese protocolo esconde. Las calificaciones tienen marca de tiempo, y en
producción el recomendador predice **lo que el usuario calificará después** de lo que ya calificó.
El split honesto para ese escenario es temporal por usuario: entrenar con sus calificaciones
antiguas, evaluar con las recientes, exactamente la lógica del corte del Lab 3.

In [ ]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"train: {train.count():,}   test: {test.count():,}")

# Películas que están en test y no en train: ALS no tiene cómo predecirlas. Este número
# explica el NaN de la sección siguiente.
items_train = train.select("movieId").distinct()
solo_test = test.select("movieId").distinct().join(items_train, "movieId", "left_anti").count()
print("películas presentes solo en test:", solo_test)

## 3. ALS y el primer error que se ve

**Qué hace ALS.** Busca dos matrices de rango `k` —una fila por usuario, una fila por película—
cuyo producto aproxime la matriz de calificaciones en las celdas conocidas. El problema conjunto
no es convexo, pero **fijada una de las matrices, la otra se obtiene con mínimos cuadrados
ordinarios, fila por fila y de forma independiente**. Por eso el algoritmo alterna (*Alternating
Least Squares*): fija usuarios y resuelve películas, fija películas y resuelve usuarios. Esas
regresiones independientes son lo que se reparte entre ejecutores, y es la razón de que MLlib
traiga ALS y no descenso de gradiente para este problema.

**Los tres hiperparámetros que importan:** `rank` (la dimensión `k`), `regParam` (regularización)
y `maxIter` (alternancias). Y una decisión que no es hiperparámetro: `coldStartStrategy`.

Por defecto vale `"nan"`: cuando un usuario o una película del test no estaba en el entrenamiento,
ALS no tiene un vector para ella y predice `NaN`. El evaluador promedia y **el RMSE completo sale
NaN**. Es el primer error del lab, y se ve a propósito.

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

# Sin coldStartStrategy: el valor por defecto es "nan".
als_nan  = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
               rank=10, regParam=0.1, maxIter=10, seed=42)
pred_nan = als_nan.fit(train).transform(test)
print("RMSE con coldStartStrategy='nan':", rmse_eval.evaluate(pred_nan))
print("filas con predicción NaN:", pred_nan.filter(F.isnan("prediction")).count())
pred_nan.filter(F.isnan("prediction")).show(3)

In [ ]:
# coldStartStrategy="drop": las filas sin predicción posible se excluyen de la evaluación.
als_base = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
               rank=10, regParam=0.1, maxIter=10, seed=42, coldStartStrategy="drop")

t0 = time.time()
modelo_base = als_base.fit(train)
seg = time.time() - t0

pred = modelo_base.transform(test)
print(f"RMSE con 'drop': {rmse_eval.evaluate(pred):.4f}   entrenamiento: {seg:.1f} s")
print(f"filas evaluadas: {pred.count():,} de {test.count():,} en test")

# Lo que ALS aprendió: un vector de 'rank' números por usuario y por película.
print("factores por usuario:", modelo_base.userFactors.count(), "| por película:", modelo_base.itemFactors.count())
modelo_base.itemFactors.show(2, truncate=90)

## 4. La función `experimento`, otra vez

La misma estructura del Lab 4 —ajustar, evaluar, cronometrar, registrar— con dos métricas propias
de un recomendador:

- **`rmse_test`**: la métrica del laboratorio. Mide qué tan bien se predice la nota.
- **`cobertura_top10`**: qué fracción del catálogo aparece al menos una vez en el top-10 de algún
  usuario. Es la medida más simple del **sesgo de popularidad**: un recomendador que le muestra a
  todos las mismas 200 películas tiene cobertura del 2% aunque su RMSE sea excelente. La métrica
  del negocio es un *ranking*, no una nota, y esta columna lo recuerda.

Y la tercera, que ya no es nueva: **`seg_entrenamiento`**, junto a las otras dos, para que la
comparación de la sección 10 pueda considerar el costo.

In [ ]:
N_CATALOGO = n_m

def cobertura_catalogo(modelo, k=10):
    """Fracción del catálogo que aparece al menos una vez en el top-k de algún usuario."""
    recs = modelo.recommendForAllUsers(k)
    distintos = recs.select(F.explode("recommendations.movieId").alias("movieId")).distinct().count()
    return distintos / N_CATALOGO

def experimento(nombre, rank, regParam, maxIter=10, datos_train=None, datos_test=None):
    """Ajusta ALS, lo evalúa y registra todo en un run de MLflow. Devuelve (ALSModel, dict)."""
    dtr = datos_train if datos_train is not None else train
    dte = datos_test  if datos_test  is not None else test

    with mlflow.start_run(run_name=nombre):
        als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
                  rank=rank, regParam=regParam, maxIter=maxIter,
                  coldStartStrategy="drop", seed=42)

        t0 = time.time()
        modelo = als.fit(dtr)
        seg_fit = time.time() - t0

        rmse = rmse_eval.evaluate(modelo.transform(dte))
        cob  = cobertura_catalogo(modelo)

        # --- Parámetros: las decisiones ---
        mlflow.log_param("tipo", "als")
        mlflow.log_param("dataset", "ml-latest-small")
        mlflow.log_param("rank", rank)
        mlflow.log_param("regParam", regParam)
        mlflow.log_param("maxIter", maxIter)
        mlflow.log_param("coldStartStrategy", "drop")

        # --- Métricas: predictiva, de negocio y computacional ---
        mlflow.log_metric("rmse_test", rmse)
        mlflow.log_metric("cobertura_top10", cob)
        mlflow.log_metric("seg_entrenamiento", seg_fit)
        mlflow.log_metric("filas_train", dtr.count())

        # --- Artefacto: el modelo exacto, recuperable ---
        mlflow.spark.log_model(modelo, "als")

        print(f"{nombre:18s} rank={rank:<3d} reg={regParam:<5}  RMSE={rmse:.4f}  cobertura={cob:.1%}  fit={seg_fit:5.1f}s")
        return modelo, {"rmse": rmse, "cobertura": cob, "seg": seg_fit}

## 5. Tres configuraciones deliberadamente distintas

| Run | Qué cambia | Qué pregunta responde |
|---|---|---|
| `als_base` | `rank=10`, `regParam=0.1` | ¿Cuánto explica un modelo pequeño? |
| `als_rank_alto` | `rank=50`, misma regularización | ¿Más dimensiones mejoran el RMSE, o solo cuestan tiempo y sobreajustan? |
| `als_reg_alta` | `rank=10`, `regParam=0.5` | ¿Qué le hace la regularización al RMSE **y a la cobertura**? |

El split, la semilla y `maxIter` se mantienen fijos: es lo que permite atribuir la diferencia a la
configuración. Cada corrida toma segundos, así que hay tiempo para agregar una cuarta propia.

In [ ]:
modelo_A, m_A = experimento("als_base",       rank=10, regParam=0.1)

In [ ]:
modelo_B, m_B = experimento("als_rank_alto",  rank=50, regParam=0.1)

In [ ]:
modelo_C, m_C = experimento("als_reg_alta",   rank=10, regParam=0.5)

In [ ]:
# Cuarta configuración, propia (opcional pero recomendable): cambiar UNA cosa y anotar por qué.
# modelo_D, m_D = experimento("als_propio", rank=20, regParam=0.05)

## 6. Lo que se recomienda, y a quién se le recomienda lo mismo

El RMSE es la métrica del laboratorio; la del negocio es un *ranking*: `recommendForAllUsers(k)`
devuelve el top-k de cada usuario. Dos lecturas sobre esa salida:

1. **Para un usuario concreto**: qué calificó alto y qué se le recomienda. La coherencia se ve a ojo.
2. **Para todos a la vez**: qué películas se repiten en los top-10. Si las más recomendadas son
   también las más calificadas, hay sesgo de popularidad; si son películas con dos o tres
   calificaciones, el problema es el inverso —recomendaciones sostenidas en poca evidencia—.
   Ambas son señales de monitoreo, y ambas se leen en la misma tabla.

In [ ]:
modelo = modelo_A
recs = modelo.recommendForAllUsers(10).cache()

USUARIO = 1
print(f"Lo que el usuario {USUARIO} calificó con 4,5 o más:")
(train.filter((F.col("userId") == USUARIO) & (F.col("rating") >= 4.5))
      .join(movies, "movieId").select("title", "genres", "rating").show(6, truncate=48))

print(f"Lo que ALS le recomienda:")
(recs.filter(F.col("userId") == USUARIO)
     .select(F.explode("recommendations").alias("r"))
     .select(F.col("r.movieId").alias("movieId"), F.round(F.col("r.rating"), 2).alias("prediccion"))
     .join(movies, "movieId").orderBy(F.desc("prediccion"))
     .select("title", "genres", "prediccion").show(10, truncate=48))

In [ ]:
# Sesgo de popularidad: qué películas se repiten en los top-10, y cuánta evidencia las sostiene.
frecuencia  = (recs.select(F.explode("recommendations.movieId").alias("movieId"))
                   .groupBy("movieId").count().withColumnRenamed("count", "veces_recomendada"))
popularidad = train.groupBy("movieId").count().withColumnRenamed("count", "n_calificaciones")

(frecuencia.join(popularidad, "movieId", "left").join(movies, "movieId")
           .orderBy(F.desc("veces_recomendada"))
           .select("title", "veces_recomendada", "n_calificaciones").show(10, truncate=48))

print(f"cobertura del catálogo en el top-10 (als_base): {m_A['cobertura']:.1%}")
print(f"cobertura del catálogo en el top-10 (als_reg_alta): {m_C['cobertura']:.1%}")

### 6b. El usuario nuevo: la pregunta que ALS no puede responder

Un usuario que llega hoy no tiene fila en la matriz, así que no tiene vector. Con
`coldStartStrategy="drop"` la respuesta no es un error: es **cero filas**. Es la forma honesta del
arranque en frío, y la segunda mitad del laboratorio existe para resolverla.

In [ ]:
NUEVO = 999_999    # un id que no existe en el entrenamiento
print("filas en recommendForAllUsers para el usuario nuevo:", recs.filter(F.col("userId") == NUEVO).count())

pedido = spark.createDataFrame([(NUEVO, 1), (NUEVO, 318)], ["userId", "movieId"])
print("filas que devuelve transform() para él:", modelo.transform(pedido).count(), "→ sin historial no hay factores, y sin factores no hay predicción")

## 7. Embeddings a escala: el modelo se carga una vez por partición

Aquí termina el aprendizaje sobre tablas y empieza el análisis de datos no estructurados. Un
*embedding* es un vector denso —384 números con `all-MiniLM-L6-v2`— que captura significado, de
modo que textos parecidos quedan cerca. Cómo se entrena ese modelo no es contenido del curso;
**cómo se aplica a millones de textos sí lo es**.

La herramienta es la **pandas UDF de tipo iterador**: recibe un iterador de lotes (`Iterator[pd.Series]`)
y devuelve otro. La diferencia con una UDF fila a fila está en la primera línea del cuerpo: el
modelo se carga **una vez por partición** y atiende todos los lotes que le llegan. Sobre 9.742
títulos la diferencia es de segundos; sobre 50 millones de descripciones de producto es la
diferencia entre 20 minutos y 20 horas.

El texto a codificar es título más géneros. Es poco —ya se verá qué alcanza a capturar—, y es
exactamente el tipo de atributo que un catálogo real siempre tiene, incluso para el producto que
se lanzó ayer y nadie ha comprado.

In [ ]:
from typing import Iterator
from pyspark.sql.functions import pandas_udf

MODELO_ST = "all-MiniLM-L6-v2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

catalogo = movies.withColumn(
    "texto", F.concat_ws(" · ", "title", F.regexp_replace("genres", r"\|", ", ")))
catalogo.select("texto").show(3, truncate=80)

def embeddings_por_particion(df):
    """El modelo se carga una vez por partición, igual que en una pandas UDF de iterador."""
    esquema = T.StructType(list(df.schema.fields) +
                           [T.StructField("embedding", T.ArrayType(T.FloatType()))])
    def por_particion(filas):
        from sentence_transformers import SentenceTransformer
        modelo_st = SentenceTransformer(MODELO_ST, device="cpu")   # una vez por partición
        import torch
        torch.set_num_threads(1)
        filas = list(filas)
        if not filas:
            return []
        vecs = modelo_st.encode([f["texto"] for f in filas], batch_size=16, normalize_embeddings=True)
        return [tuple(f) + ([float(x) for x in v],) for f, v in zip(filas, vecs)]
    return spark.createDataFrame(df.rdd.mapPartitions(por_particion), esquema)

In [ ]:
# En macOS local se usa una partición para evitar que PyTorch abra varios workers nativos.
# En Colab o en un cluster se puede volver a repartition(2) o aumentar este número.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

particiones_embeddings = 1 if platform.system() == "Darwin" else 2
t0 = time.time()
emb = embeddings_por_particion(catalogo.repartition(particiones_embeddings)).cache()
n_emb = emb.count()
seg_emb = time.time() - t0
print(f"{n_emb:,} embeddings en {seg_emb:.0f} s (incluye cargar el modelo en cada partición)")

emb.select("title", F.size("embedding").alias("dim"),
           F.slice("embedding", 1, 4).alias("primeros_4")).show(3, truncate=60)

# El costo también se registra: la tabla de la sección 10 lo muestra junto a los de ALS.
with mlflow.start_run(run_name="embeddings_minilm"):
    mlflow.log_param("tipo", "contenido")
    mlflow.log_param("modelo", MODELO_ST)
    mlflow.log_param("dim", 384)
    mlflow.log_metric("seg_embeddings", seg_emb)
    mlflow.log_metric("n_items", n_emb)

## 8. Búsqueda semántica: similitud coseno sobre el catálogo

Los vectores se normalizaron al codificar, así que la **similitud coseno es un producto punto**.
Con 9.742 × 384 números la matriz completa cabe en memoria del driver y una consulta es una
multiplicación de matrices; a escala real esa multiplicación se reemplaza por un **índice
vectorial** (FAISS en local, o una base vectorial como las vistas en Big Data y Cloud Computing).
El puente queda como extensión E3; la lógica es la misma.

In [ ]:
pdf = emb.select("movieId", "title", "genres", "embedding").toPandas()
MATRIZ  = np.vstack(pdf["embedding"].to_list()).astype(np.float32)     # 9.742 × 384
IDS     = pdf["movieId"].to_numpy()
TITULOS = pdf["title"].to_numpy()
GENEROS = pdf["genres"].to_numpy()
print(MATRIZ.shape, "· normas ≈ 1:", np.round(np.linalg.norm(MATRIZ[:3], axis=1), 3))

from sentence_transformers import SentenceTransformer
st_driver = SentenceTransformer(MODELO_ST)      # una copia en el driver, solo para codificar consultas

def buscar(consulta, k=5):
    """Top-k del catálogo más cercano a una frase libre."""
    q = st_driver.encode([consulta], normalize_embeddings=True)[0]
    sims = MATRIZ @ q
    idx = np.argsort(-sims)[:k]
    return pd.DataFrame({"movieId": IDS[idx], "titulo": TITULOS[idx],
                         "generos": GENEROS[idx], "similitud": sims[idx].round(3)})

def similares(movie_id, k=5):
    """Top-k de películas vecinas a una dada, por contenido."""
    i = int(np.where(IDS == movie_id)[0][0])
    sims = MATRIZ @ MATRIZ[i]
    idx = np.argsort(-sims)[1:k + 1]          # excluye la propia película
    return pd.DataFrame({"titulo": TITULOS[idx], "generos": GENEROS[idx], "similitud": sims[idx].round(3)})

buscar("space adventure with robots and a rebellion")

In [ ]:
# La favorita del usuario 1 según ALS (sección 6) y sus vecinas por contenido: las dos vías
# sobre las mismas películas. Comparar esta lista con la que ALS le recomendó.
fav = (train.filter(F.col("userId") == USUARIO)
            .orderBy(F.desc("rating"), F.desc("timestamp")).first()).movieId
print("favorita del usuario", USUARIO, "→", TITULOS[IDS == fav][0])
similares(fav)

## 9. El usuario nuevo, resuelto por contenido

El mismo usuario para el que ALS devolvió cero filas. Basta **una frase suya** —lo que busca, o el
título de una película que le gustó— para tener un top-5 razonable, porque el contenido del
catálogo existe desde el día uno aunque nadie lo haya calificado.

Esto es **recomendación por contenido** (*content-based*), y en producción convive con ALS: la
primera cubre el arranque en frío y el surtido nuevo; la segunda aprende lo que el contenido no
dice —qué películas se ven juntas aunque no se parezcan—. El sistema resultante se llama
**híbrido**, y ese es el criterio de decisión de la sesión:

- **ALS** cuando abunda la interacción.
- **Embeddings** cuando hay texto o atributos y poca interacción.
- **Híbrido** cuando hay ambos, que es casi siempre.

Una nota honesta sobre el modelo: `all-MiniLM-L6-v2` fue entrenado sobre texto en inglés y el
catálogo está en inglés, así que las consultas en inglés funcionan mejor. La variante multilingüe
queda en la extensión E4.

In [ ]:
buscar("dark psychological thriller with an unreliable narrator")

In [ ]:
# ENTREGA · pieza 2: escribir aquí una consulta propia. La tabla que devuelve es parte de la entrega.
MI_CONSULTA = "Ciberpunk"
buscar(MI_CONSULTA)

## 10. `search_runs()`: la tabla comparativa — **este es el entregable**

Tercera vez que se lee un experimento como consulta y no como recuerdo. La tabla muestra las
configuraciones de ALS con sus tres métricas —RMSE, cobertura y tiempo— y, en la última fila, el
costo de calcular los embeddings. La comparación que se pide no es "cuál tiene el RMSE más bajo",
sino **cuál conviene, y para qué usuario**.

In [ ]:
pd.set_option("display.width", 180)

runs = mlflow.search_runs(order_by=["metrics.rmse_test ASC"])

COLS = ["tags.mlflow.runName", "params.tipo", "params.rank", "params.regParam", "params.maxIter",
        "metrics.rmse_test", "metrics.cobertura_top10", "metrics.seg_entrenamiento",
        "params.modelo", "metrics.seg_embeddings"]
tabla = runs[[c for c in COLS if c in runs.columns]].copy()
tabla.columns = [c.split(".")[-1] for c in tabla.columns]
tabla = tabla.rename(columns={"runName": "experimento"}).round(4)

print(f"runs registrados: {len(runs)}")
tabla

In [ ]:
# Exportar la tabla: es el archivo que se adjunta en Canvas.
tabla.to_csv("lab5_comparacion_als.csv", index=False)
print(tabla.to_markdown(index=False))

### 10b. Párrafo de lectura — **respuesta final (se evalúa)**

Reemplazar el texto entre corchetes por la lectura propia. Un párrafo, entre cinco y ocho líneas,
que responda las tres preguntas:

1. **Qué configuración de ALS conviene** y por qué: la diferencia de RMSE en cifras, lo que pasó
   con la cobertura y cuánto costó cada corrida. Si el `rank` alto mejoró poco y tardó mucho,
   decirlo.
2. **Qué le recomendaría ALS al usuario nuevo** de la sección 6b, y por qué.
3. **Cómo cubre ese caso la búsqueda por contenido**, con el top-5 de la consulta propia como
   evidencia, y qué combinación se propondría en producción.

Elegiría `als_rank_alto` (`rank=50`, `regParam=0,1`): reduce el RMSE de 0,8814 a 0,8736 —una mejora pequeña de 0,0078— y aumenta la cobertura del 6,83 % al 12,95 %, aunque tarda 2,3041 s frente a 1,4534 s del baseline, aproximadamente 1,59 veces más tiempo.
La configuración `als_reg_alta` tarda 1,4713 s, pero empeora el RMSE a 0,9978 y reduce la cobertura al 0,32 %, por lo que priorizaría el rango alto por su mayor cobertura y menor error.
Para el usuario nuevo `999999` de la sección 6b, ALS devuelve cero filas porque no dispone de factores aprendidos; `coldStartStrategy="drop"` excluye las predicciones indefinidas, pero no resuelve el arranque en frío.
La consulta «Ciberpunk» devuelve *Marauders*, *Cirque du Freak: The Vampire’s Assistant*, *Jim Gaffigan: Cinco*, *Incognito* y *The Smurfs*, con similitudes entre 0,339 y 0,388, sin necesitar historial.
Ese top-5 no muestra una correspondencia temática clara: disponer de candidatos no garantiza su pertinencia; la extensión multilingüe devuelve, entre otros, *Hackers* y *Hardware*, lo que sugiere una mejora cualitativa que aún requiere evaluar relevancia.
En producción propondría contenido para usuarios nuevos según preferencias declaradas y ALS cuando exista historial, vigilando idioma, relevancia del ranking, cobertura y costo.

**Lo que hace fuerte esta entrega.** Que la elección esté sostenida por las tres columnas de la
tabla —RMSE, cobertura y tiempo— y no solo por la primera; que las cifras citadas coincidan con
las registradas en MLflow; y que el párrafo nombre el arranque en frío con el ejemplo concreto de
la sección 6b, no en abstracto.

## 11. Ejercicios de extensión (propuestos, no evaluados)

**E1 · Split temporal por usuario.** Para cada usuario, las calificaciones más recientes (por
`timestamp`) van a test y el resto a train (`row_number` sobre una ventana por `userId` ordenada
por tiempo, como en la sesión 2). Comparar el RMSE con el de `randomSplit` y explicar la
diferencia con la misma lógica del corte del Lab 3.

**E2 · Retroalimentación implícita.** En retail no hay nota: hay clics y compras. Convertir la
calificación en interacción binaria (`rating >= 4`), entrenar con `implicitPrefs=True` y evaluar
con `precision@k` sobre `recommendForAllUsers` —el RMSE deja de tener sentido y conviene entender
por qué—.

**E3 · Índice vectorial.** `pip install faiss-cpu`; `indice = faiss.IndexFlatIP(384)`;
`indice.add(MATRIZ)`; `indice.search(q[None, :], 5)`. Mismo resultado que el producto punto; la
diferencia aparece con millones de vectores, donde un índice aproximado (`IndexIVFFlat`,
`IndexHNSWFlat`) cambia el orden de magnitud del tiempo de consulta.

**E4 · Modelo multilingüe.** Repetir la sección 8 con `paraphrase-multilingual-MiniLM-L12-v2` y
consultar en español. Medir el tiempo de embedding contra el registrado: el modelo es más grande.

**E5 · La escala, para quien quiera medirla.** MovieLens 25M
(`https://files.grouplens.org/datasets/movielens/ml-25m.zip`, 250 MB; 162 mil usuarios, 62 mil
películas, 25 millones de calificaciones). Correr solo `als_base` y registrarlo con la misma
función `experimento`: la fila queda en la misma tabla y la columna `seg_entrenamiento` cuenta
la historia. En Colab con dos núcleos son varios minutos por corrida; ese es el punto.

---

### E1 · Solución: split temporal por usuario

Se reserva aproximadamente el 20 % más reciente de cada usuario, manteniendo al menos una observación en entrenamiento. Los empates de timestamp se desempatan por movieId: se comprueba `max(train) <= min(test)`, no una separación temporal estricta entre empates. Este protocolo limita el uso de futuro **dentro de cada usuario**; no equivale a un corte global del calendario.

Se conserva rank=10, regParam=0,1 y maxIter=10. El RMSE temporal se compara con el baseline de la sesión y se informa la fracción evaluada tras `drop`. Puede aumentar o disminuir: no se afirma el signo sin ejecutar. Un aumento sería compatible con una tarea futura más difícil; no demuestra por sí solo sobreajuste.

In [ ]:
EJECUTAR_E1 = True
if EJECUTAR_E1:
    from pyspark.sql.window import Window
    w_usuario = Window.partitionBy("userId")
    w_tiempo = w_usuario.orderBy(F.col("timestamp"), F.col("movieId"))
    ordenadas = (ratings.withColumn("n", F.count("*").over(w_usuario))
                .withColumn("pos", F.row_number().over(w_tiempo))
                .withColumn("corte", F.greatest(F.lit(1), F.floor(F.col("n") * 0.8))))
    train_temporal = ordenadas.filter("pos <= corte").select(*ratings.columns).cache()
    test_temporal = ordenadas.filter("pos > corte").select(*ratings.columns).cache()
    assert train_temporal.count() + test_temporal.count() == ratings.count()
    limites = (train_temporal.groupBy("userId").agg(F.max("timestamp").alias("fin_train"))
               .join(test_temporal.groupBy("userId").agg(F.min("timestamp").alias("inicio_test")), "userId"))
    assert limites.filter("fin_train > inicio_test").count() == 0
    modelo_temporal, metricas_temporales = experimento(
        "e1_als_temporal", rank=10, regParam=0.1,
        datos_train=train_temporal, datos_test=test_temporal)
    n_test_temporal = test_temporal.count()
    n_eval_temporal = modelo_temporal.transform(test_temporal).count()
    display(pd.DataFrame([
        {"split": "aleatorio", **m_A,
         "fraccion_evaluada": modelo_A.transform(test).count() / test.count()},
        {"split": "temporal_usuario", **metricas_temporales,
         "fraccion_evaluada": n_eval_temporal / n_test_temporal}]))
    print("Diferencia RMSE temporal - aleatorio:", metricas_temporales["rmse"] - m_A["rmse"])
    train_temporal.unpersist(); test_temporal.unpersist()
else:
    print("E1 pendiente: activa EJECUTAR_E1 para medir el split temporal.")


### E2 · Solución: interacciones implícitas y precision@10

`rating >= 4` define la relevancia positiva. Se conservan las interacciones positivas de train con valor 1; las demás no son señales positivas. Este es un ejercicio de binarización de notas, no evidencia real de compras. Con `implicitPrefs=True`, ALS aprende preferencias con confianza (`alpha=1` explícito), no predice notas en la escala 0,5–5, por lo que el RMSE de notas deja de ser la evaluación adecuada.

Se generan candidatos mediante `recommendForAllUsers`, se excluyen todas las películas vistas en train y se seleccionan diez por usuario. Se evalúan usuarios conocidos con al menos un positivo en test que pertenezca al catálogo aprendido. `precision@10 = aciertos / 10`; las listas cortas conservan denominador 10. También se reportan usuarios elegibles y fracción de positivos recuperables: la métrica es condicional a ese universo.

Para garantizar diez no vistos cuando existan, el número de candidatos incluye el máximo de películas vistas por usuario. Es viable para MovieLens pequeño, no una estrategia de recuperación para catálogos masivos.

In [ ]:
EJECUTAR_E2 = True
if EJECUTAR_E2:
    from pyspark.sql.window import Window
    k = 10
    train_imp = (train.filter(F.col("rating") >= 4)
                 .select("userId", "movieId").distinct().withColumn("interaccion", F.lit(1.0)))
    t0_imp = time.perf_counter()
    modelo_imp = ALS(userCol="userId", itemCol="movieId", ratingCol="interaccion",
                     rank=10, regParam=0.1, maxIter=10, seed=42,
                     implicitPrefs=True, alpha=1.0, coldStartStrategy="drop").fit(train_imp)
    segundos_imp = time.perf_counter() - t0_imp
    usuarios_imp = modelo_imp.userFactors.select(F.col("id").alias("userId"))
    items_imp = modelo_imp.itemFactors.select(F.col("id").alias("movieId"))
    positivos_test = test.filter(F.col("rating") >= 4).select("userId", "movieId").distinct()
    relevantes = positivos_test.join(usuarios_imp, "userId").join(items_imp, "movieId")
    elegibles = relevantes.select("userId").distinct()
    vistos = train.select("userId", "movieId").distinct()
    max_vistos = vistos.groupBy("userId").count().agg(F.max("count")).first()[0]
    candidatos_k = min(items_imp.count(), k + max_vistos)
    candidatos = (modelo_imp.recommendForAllUsers(candidatos_k)
                  .select("userId", F.explode("recommendations").alias("r"))
                  .select("userId", F.col("r.movieId").alias("movieId"), F.col("r.rating").alias("score"))
                  .join(vistos, ["userId", "movieId"], "left_anti"))
    orden = Window.partitionBy("userId").orderBy(F.desc("score"), "movieId")
    top_imp = candidatos.withColumn("pos", F.row_number().over(orden)).filter(F.col("pos") <= k)
    aciertos = (top_imp.join(relevantes, ["userId", "movieId"])
                .groupBy("userId").count().withColumnRenamed("count", "aciertos"))
    por_usuario = elegibles.join(aciertos, "userId", "left").fillna(0, subset=["aciertos"])
    precision10 = por_usuario.select((F.col("aciertos") / k).alias("p")).agg(F.avg("p")).first()[0]
    assert precision10 is not None and 0 <= precision10 <= 1
    fraccion_recuperable = relevantes.count() / positivos_test.count()
    with mlflow.start_run(run_name="e2_als_implicito"):
        mlflow.log_params({"tipo": "als_implicito", "dataset": "ml-latest-small", "rank": 10,
                           "regParam": 0.1, "maxIter": 10, "alpha": 1.0, "k": k})
        mlflow.log_metrics({"precision_at_10": precision10, "seg_entrenamiento": segundos_imp,
                            "usuarios_elegibles": elegibles.count(),
                            "fraccion_positivos_recuperables": fraccion_recuperable})
    print("Precision@10:", precision10, "Usuarios evaluados:", elegibles.count())
    print("Fracción de positivos de test recuperables:", fraccion_recuperable)
else:
    print("E2 pendiente: activa EJECUTAR_E2 para evaluar el ranking implícito.")


### E3 · Solución: búsqueda exacta con FAISS

Instala `faiss-cpu` en el intérprete del kernel desde la terminal de VS Code (`python -m pip install faiss-cpu`). No se instala automáticamente para no cambiar el entorno validado. Usa vectores float32 contiguos y normalizados. `IndexFlatIP` sigue siendo búsqueda exacta, no aproximada; debe coincidir en puntajes con NumPy dentro de tolerancia numérica. En empates pueden cambiar los identificadores seleccionados u ordenados. Se separa construir el índice de consultar; se comparan medianas tras calentamiento. No se presupone que FAISS sea más rápido con 9.742 vectores.

In [ ]:
EJECUTAR_E3 =True
if EJECUTAR_E3:
    import faiss
    X_faiss = np.ascontiguousarray(MATRIZ, dtype=np.float32)
    q_faiss = np.ascontiguousarray(st_driver.encode([MI_CONSULTA], normalize_embeddings=True), dtype=np.float32)
    assert X_faiss.shape[1] == q_faiss.shape[1] == 384
    t0_idx = time.perf_counter()
    indice = faiss.IndexFlatIP(384)
    indice.add(X_faiss)
    seg_indice = time.perf_counter() - t0_idx
    D, I = indice.search(q_faiss, 5)
    sims_np = X_faiss @ q_faiss[0]
    idx_np = np.argsort(-sims_np)[:5]
    assert np.allclose(D[0], sims_np[idx_np], atol=1e-5)
    def medir_mediana(funcion, repeticiones=20):
        funcion()
        tiempos = []
        for _ in range(repeticiones):
            inicio = time.perf_counter(); funcion()
            tiempos.append(time.perf_counter() - inicio)
        return float(np.median(tiempos))
    seg_numpy = medir_mediana(lambda: np.argsort(-(X_faiss @ q_faiss[0]))[:5])
    seg_faiss = medir_mediana(lambda: indice.search(q_faiss, 5))
    display(pd.DataFrame({"titulo": TITULOS[I[0]], "similitud": D[0]}))
    print({"seg_construccion_indice": seg_indice,
           "mediana_consulta_numpy_s": seg_numpy, "mediana_consulta_faiss_s": seg_faiss,
           "mismo_conjunto_top5": set(I[0]) == set(idx_np)})
else:
    print("E3 pendiente: instala FAISS en tu entorno y activa EJECUTAR_E3.")


### E4 · Solución: modelo multilingüe y consulta en español

Se recrean los embeddings del catálogo con `paraphrase-multilingual-MiniLM-L12-v2` y se codifica la consulta con **ese mismo modelo**. No se mezcla una consulta multilingüe con la matriz MiniLM inglesa: aunque ambas tengan 384 dimensiones, son espacios diferentes.

Se conservan particiones y batch_size=64 del original para aproximar una comparación de tiempos; el dispositivo se fija en CPU en la nueva prueba y se registra. El primer tiempo puede incluir descarga del modelo y cachés distintas, por lo que la razón respecto a `seg_emb` es descriptiva, no un benchmark controlado. Examina títulos y géneros del nuevo top-5 y declara si mejora su pertinencia; tener soporte multilingüe no garantiza una buena respuesta sobre ciberpunk con metadatos tan breves.

In [ ]:
EJECUTAR_E4 =True
if EJECUTAR_E4:
    MODELO_MULTI = "paraphrase-multilingual-MiniLM-L12-v2"
    particiones_multi = emb.rdd.getNumPartitions()
    def crear_embeddings_multi(df):
        from itertools import islice
        esquema_multi = T.StructType(list(df.schema.fields) +
                        [T.StructField("embedding", T.ArrayType(T.FloatType()))])
        def transformar(filas):
            from sentence_transformers import SentenceTransformer
            modelo_local = SentenceTransformer(MODELO_MULTI, device="cpu")
            it = iter(filas)
            while True:
                lote = list(islice(it, 128))
                if not lote:
                    break
                vectores = modelo_local.encode([r["texto"] for r in lote], batch_size=64,
                            normalize_embeddings=True, show_progress_bar=False)
                for fila, vector in zip(lote, vectores):
                    yield tuple(fila) + ([float(x) for x in vector],)
        return spark.createDataFrame(df.rdd.mapPartitions(transformar), esquema_multi)
    t0_multi = time.perf_counter()
    emb_multi = crear_embeddings_multi(catalogo.repartition(particiones_multi)).cache()
    n_multi = emb_multi.count()
    seg_multi = time.perf_counter() - t0_multi
    assert n_multi == n_m
    pdf_multi = emb_multi.select("movieId", "title", "genres", "embedding").toPandas()
    matriz_multi = np.vstack(pdf_multi["embedding"].tolist()).astype(np.float32)
    consulta_es = "Ciberpunk"
    modelo_multi_driver = SentenceTransformer(MODELO_MULTI, device="cpu")
    q_multi = modelo_multi_driver.encode([consulta_es], normalize_embeddings=True)[0]
    sims_multi = matriz_multi @ q_multi
    idx_multi = np.argsort(-sims_multi)[:5]
    top_multi = pdf_multi.iloc[idx_multi][["movieId", "title", "genres"]].copy()
    top_multi["similitud"] = sims_multi[idx_multi]
    display(top_multi)
    with mlflow.start_run(run_name="e4_embeddings_multilingue"):
        mlflow.log_params({"tipo": "contenido", "modelo": MODELO_MULTI, "dispositivo": "cpu",
                           "particiones": particiones_multi, "batch_size": 64, "consulta": consulta_es})
        mlflow.log_metrics({"seg_embeddings": seg_multi, "n_items": n_multi})
    print("Segundos original / multilingüe:", seg_emb, seg_multi,
          "| razón descriptiva:", seg_multi / seg_emb)
    emb_multi.unpersist()
else:
    print("E4 pendiente: activa EJECUTAR_E4; requiere descargar un modelo adicional.")


### E5 · Solución: escalar a MovieLens 25M

Descarga el ZIP oficial indicado en el enunciado y extrae `ratings.csv` y `movies.csv` en una carpeta `ml-25m` junto al notebook (o cambia `RUTA_25M`). La celda no descarga 25 millones de filas automáticamente. El flag es una protección operativa; no representa resultados ya obtenidos.

Se ejecuta únicamente el baseline. Se reutiliza el cuerpo de `experimento` con dos correcciones explícitas en una variante local: etiqueta de dataset `ml-25m` y denominador de cobertura propio de 25M. Usar sin ajustes la función original registraría un nombre de dataset incorrecto y dividiría por 9.742 películas. Los datos pequeños y sus modelos permanecen intactos.

El tiempo guardado corresponde solo a `fit`; calcular cobertura para todos los usuarios y guardar el modelo añaden trabajo fuera de ese cronómetro. Un Mac puede quedarse sin memoria al cachear y entrenar con 25M: si ocurre, registra el fallo y recursos, no un tiempo inventado. Se usa almacenamiento MEMORY_AND_DISK para permitir derrame. Reporta hardware, filas, particiones, versiones, RMSE, cobertura y tiempo. Una diferencia con el dataset pequeño refleja escala y cambio de población; no demuestra aceleración distribuida por sí sola.

In [ ]:
EJECUTAR_E5 = True
if EJECUTAR_E5:
    from pathlib import Path
    from pyspark import StorageLevel
    RUTA_25M = Path.cwd() / "ml-25m"
    if not all((RUTA_25M / nombre).is_file() for nombre in ("ratings.csv", "movies.csv")):
        raise FileNotFoundError("No se encontró MovieLens 25M. Descarga el ZIP oficial, extrae la carpeta ml-25m y ajusta RUTA_25M.")
    schema_25 = "userId INT, movieId INT, rating DOUBLE, timestamp LONG"
    ratings_25 = spark.read.schema(schema_25).option("header", True).csv(str(RUTA_25M / "ratings.csv"))
    movies_25 = spark.read.option("header", True).option("inferSchema", True).csv(str(RUTA_25M / "movies.csv"))
    assert ratings_25.count() == 25000095, "Comprueba que sea la versión oficial MovieLens 25M."
    train_25, test_25 = ratings_25.randomSplit([0.8, 0.2], seed=42)
    train_25.persist(StorageLevel.MEMORY_AND_DISK); test_25.persist(StorageLevel.MEMORY_AND_DISK)
    print("Filas train/test:", train_25.count(), test_25.count())
    N_CATALOGO_25M = movies_25.count()

    def cobertura_catalogo_25m(modelo, k=10):
        """Fracción del catálogo que aparece al menos una vez en el top-k de algún usuario."""
        recs = modelo.recommendForAllUsers(k)
        distintos = recs.select(F.explode("recommendations.movieId").alias("movieId")).distinct().count()
        return distintos / N_CATALOGO_25M

    def experimento_25m(nombre, rank, regParam, maxIter=10, datos_train=None, datos_test=None):
        """Ajusta ALS, lo evalúa y registra todo en un run de MLflow. Devuelve (ALSModel, dict)."""
        dtr = datos_train if datos_train is not None else train
        dte = datos_test if datos_test is not None else test

        with mlflow.start_run(run_name=nombre):
            als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
                      rank=rank, regParam=regParam, maxIter=maxIter,
                      coldStartStrategy="drop", seed=42)

            t0 = time.time()
            modelo = als.fit(dtr)
            seg_fit = time.time() - t0

            rmse = rmse_eval.evaluate(modelo.transform(dte))
            cob  = cobertura_catalogo_25m(modelo)

            mlflow.log_param("tipo", "als")
            mlflow.log_param("dataset", "ml-25m")
            mlflow.log_param("rank", rank)
            mlflow.log_param("regParam", regParam)
            mlflow.log_param("maxIter", maxIter)
            mlflow.log_param("coldStartStrategy", "drop")
            mlflow.log_metric("rmse_test", rmse)
            mlflow.log_metric("cobertura_top10", cob)
            mlflow.log_metric("seg_entrenamiento", seg_fit)
            mlflow.log_metric("filas_train", dtr.count())
            mlflow.spark.log_model(modelo, "als")

            print(f"{nombre:18s} rank={rank:<3d} reg={regParam:<5}  RMSE={rmse:.4f}  cobertura={cob:.1%}  fit={seg_fit:5.1f}s")
            return modelo, {"rmse": rmse, "cobertura": cob, "seg": seg_fit}
    try:
        modelo_25, metricas_25 = experimento_25m("e5_als_base_25m", rank=10, regParam=0.1,
                                              datos_train=train_25, datos_test=test_25)
        print("Razón de tiempos fit 25M / small:", metricas_25["seg"] / m_A["seg"])
        display(mlflow.search_runs(filter_string="params.dataset = 'ml-25m'"))
    finally:
        train_25.unpersist(); test_25.unpersist()
else:
    print("E5 omitida: MovieLens 25M no está descargado. Descarga y extrae ml-25m, ajusta RUTA_25M y cambia EJECUTAR_E5 a True para ejecutarla.")

## Declaración de uso de IA

| Sección | Herramienta | Instrucción usada | Cómo se validó |
|---|---|---|---|
| Entorno y datos | ChatGPT / Codex | Adaptar Colab a VS Code y diagnosticar el catálogo incompleto. | Se revisaron código y datos; Dante ejecutó el notebook corregido y se conservaron sus salidas. |
| Entrega y 10b | ChatGPT / Codex | Comparar RMSE, cobertura y tiempo y explicar el top-5 y cold start. | Se contrastó la redacción con las salidas y las últimas tres corridas base finalizadas de MLflow, identificadas por run_id. |
| E1–E5 | ChatGPT / Codex | Desarrollar extensiones explicadas. | Dante ejecutó las cinco extensiones. Se revisaron sus resultados guardados; E1, E2, E4 y E5 se contrastaron con MLflow y E3 con la salida del notebook. |
| Actualización documental | ChatGPT / Codex | Sincronizar notebook, HTML y registros completos. | Consulta de SQLite en solo lectura, conservación del código y salidas y verificación de coherencia de exportaciones; no se reentrenó. |

## Cierre de aprendizaje

1. Contenido: ALS, cold start, búsqueda vectorial, evaluación temporal, feedback implícito y escala.
2. Conocimientos previos: DataFrames, ventanas SQL, álgebra lineal, métricas y MLflow.
3. Comprensión: error de notas, calidad de ranking, cobertura y tiempo miden dimensiones distintas.
4. Práctica: laboratorio base y cinco extensiones ejecutadas por Dante.
5. Confusiones corregidas: similitud no es probabilidad; drop no resuelve cold start; no deben mezclarse corridas sin identificar su origen.
6. Evidencia: 18 runs registrados; E3 documentado por separado; comparación de modelos y consultas con resultados guardados.
7. Regla: conservar parámetros, run_id, protocolo y universo de evaluación junto a cada conclusión.
8. Pendientes: revisión personal de la interpretación y declaración antes de entregar; no hay extensiones pendientes de ejecución en esta copia.
9. Siguiente ejercicio: evaluar relevancia con criterios explícitos y repetir tiempos bajo condiciones comparables.


## Lectura de las extensiones ejecutadas

- **E1:** RMSE temporal 0,933036 frente a 0,881383 aleatorio; aumenta 0,051653. La salida informa fracciones evaluadas de 0,916638 y 0,959473, respectivamente: además del protocolo cambia la población predecible tras `drop`. No se atribuye toda la diferencia a una sola causa.
- **E2:** precision@10 = 0,205743 para 592 usuarios elegibles (aproximadamente 2,06 aciertos entre diez recomendaciones por usuario). El 93,66 % de los positivos de test es recuperable bajo el universo definido. No es comparable al RMSE de notas.
- **E3:** FAISS y NumPy devuelven el mismo conjunto top-5. Las medianas observadas fueron 0,000724146 s y 0,0064355415 s; construcción del índice: 0,018305 s. En esa prueba FAISS resultó aproximadamente 8,89 veces más rápido, sin demostrar ese factor a otra escala. IndexFlatIP es exacto.
- **E4:** el modelo multilingüe devuelve Hackers, Lawnmower Man 2: Beyond Cyberspace, Theremin: An Electronic Odyssey, Webmaster y Hardware. Parece más pertinente para ciberpunk, aunque no todos los candidatos son claramente relevantes. Crear embeddings tardó 47,7835 s frente a 35,9894 s del modelo original (1,33 veces); no es una comparación controlada de cachés y descargas.
- **E5:** train contiene 19.998.209 filas y test 5.001.886. RMSE = 0,806265, cobertura = 2,5359 %, ajuste = 27,8378 s (19,15 veces el baseline pequeño). El cambio de dataset también cambia usuarios e ítems; menor RMSE no demuestra que aumentar el volumen siempre mejore el modelo. El cronómetro no incluye evaluación, cobertura ni guardado.

Fuentes: runs identificados en `lab5_registro_completo.csv` y salidas preservadas del notebook. E3 solo tiene evidencia en el notebook; no se creó retroactivamente un run para simular tracking inexistente.


## Registro completo al cierre — exportación de MLflow

Lectura posterior de la base SQLite; no sustituye la salida histórica de la sección 10. Los blancos corresponden a métricas no aplicables o no registradas, no a cero.

| run_id                           | experimento               | dataset         | rmse_test   | cobertura_top10   | seg_entrenamiento   | seg_embeddings   | precision_at_10   | start_time_utc                   |
|:---------------------------------|:--------------------------|:----------------|:------------|:------------------|:--------------------|:-----------------|:------------------|:---------------------------------|
| 7ce8e3eae3e143fd9e5d265c110ba3d5 | als_base                  | ml-latest-small | 0.881383    | 0.068261          | 2.12141             |                  |                   | 2026-09-18 14:19:34.244000+00:00 |
| ae319ad1f628425b82d7ea37cface198 | als_rank_alto             | ml-latest-small | 0.873574    | 0.129542          | 4.228529            |                  |                   | 2026-09-18 14:19:59.305000+00:00 |
| de1c84e3c17d4bf594988f77373aedb2 | als_reg_alta              | ml-latest-small | 0.997777    | 0.003182          | 1.738118            |                  |                   | 2026-09-18 14:20:23.019000+00:00 |
| 8a079235c29e4ba38c183e1510f2c5be | embeddings_minilm         |                 |             |                   |                     | 29.946975        |                   | 2026-09-18 14:22:38.306000+00:00 |
| aac788439afb45b3bc89ed91246d2df2 | als_base                  | ml-latest-small | 0.881383    | 0.068261          | 1.717823            |                  |                   | 2026-09-18 21:43:10.269000+00:00 |
| b76b4b946e354e499e2c558d857549bb | als_rank_alto             | ml-latest-small | 0.873574    | 0.129542          | 3.030534            |                  |                   | 2026-09-18 21:43:27.102000+00:00 |
| 06993e43d8524ff4bd5c122a816b1a12 | als_reg_alta              | ml-latest-small | 0.997777    | 0.003182          | 1.789549            |                  |                   | 2026-09-18 21:43:41.942000+00:00 |
| ac321b27f78342a499805cdf8bb22755 | embeddings_minilm         |                 |             |                   |                     | 23.678037        |                   | 2026-09-18 21:44:21.043000+00:00 |
| 93975a0b4a6e4769bcc06ddeb9297656 | e1_als_temporal           | ml-latest-small | 0.933036    | 0.066208          | 1.755131            |                  |                   | 2026-09-18 21:48:53.816000+00:00 |
| 9310f4189f664f9a81e396a3855fbfc5 | e2_als_implicito          | ml-latest-small |             |                   | 2.426427            |                  | 0.205743          | 2026-09-18 21:49:50.599000+00:00 |
| ba9f050134b14ae991590295e2937068 | als_base                  | ml-latest-small | 0.881383    | 0.068261          | 1.453365            |                  |                   | 2026-09-18 21:53:00.483000+00:00 |
| 9fbf0f8caf9b463bb36ace59e72001a0 | als_rank_alto             | ml-latest-small | 0.873574    | 0.129542          | 2.304079            |                  |                   | 2026-09-18 21:53:16.541000+00:00 |
| dd46f64db2bf4f148066df10a084a37d | als_reg_alta              | ml-latest-small | 0.997777    | 0.003182          | 1.471338            |                  |                   | 2026-09-18 21:53:29.601000+00:00 |
| bf42b2c04fe244248d06e20c52f5d346 | embeddings_minilm         |                 |             |                   |                     | 35.989356        |                   | 2026-09-18 21:54:19.476000+00:00 |
| bb56722ff9624879a0e1b4c2fdaf1662 | e1_als_temporal           | ml-latest-small | 0.933036    | 0.066208          | 1.444218            |                  |                   | 2026-09-18 21:54:33.601000+00:00 |
| c11de550b0fb4b6ca6ffe43e561411c2 | e2_als_implicito          | ml-latest-small |             |                   | 2.134407            |                  | 0.205743          | 2026-09-18 21:54:57.940000+00:00 |
| 143af6f8ac7b4ed6a45a2419fa6dee52 | e4_embeddings_multilingue |                 |             |                   |                     | 47.783535        |                   | 2026-09-18 21:56:15.884000+00:00 |
| cb0b87607a454c85a2d86d94dd0b663f | e5_als_base_25m           | ml-25m          | 0.806265    | 0.025359          | 27.837784           |                  |                   | 2026-09-18 21:59:56.240000+00:00 |